In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString
import os


In [2]:
print(os.listdir('../../data/hcm_data'))
print(os.listdir('../../data/hcm_data/nwk_hcm'))

['nwk_hcm', 'test.npy', 'train.npy', 'val.npy']
['hcm_edges.cpg', 'hcm_edges.dbf', 'hcm_edges.prj', 'hcm_edges.shp', 'hcm_edges.shx', 'hcm_edges_new_simplify.pkl', 'hcm_nodes.cpg', 'hcm_nodes.dbf', 'hcm_nodes.prj', 'hcm_nodes.shp', 'hcm_nodes.shx', 'hcm_nodes_new.pkl', 'hcm_pois.cpg', 'hcm_pois.dbf', 'hcm_pois.prj', 'hcm_pois.shp', 'hcm_pois.shx', 'ubodt.txt']


In [3]:
import pickle
data_path = "../../data/hcm_data/"
with open(os.path.join(data_path,"nwk_hcm/hcm_edges_new_simplify.pkl"), 'rb') as f:
    edgeinfo = pickle.load(f)
with open(os.path.join(data_path,"nwk_hcm/hcm_nodes_new.pkl"), 'rb') as f:
    nodeinfo = pickle.load(f)

In [4]:
import osmnx as ox
import networkx as nx

In [5]:
min_lat, min_lon, max_lat, max_lon = float('inf'), float('inf'), float('-inf'), float('-inf')
pad = 0.01  # Add a small padding to ensure we capture all nodes
for _, (lon, lat, _) in nodeinfo.items():
    min_lat = min(min_lat, lat)
    max_lat = max(max_lat, lat)
    min_lon = min(min_lon, lon)
    max_lon = max(max_lon, lon)

print(f"Bounding box: ({min_lat}, {min_lon}), ({max_lat}, {max_lon})")

bbox = (min_lon - pad, min_lat - pad, max_lon + pad, max_lat + pad)

Bounding box: (10.7301793, 106.6164137), (10.8524436, 106.7132954)


Dates with existing ways: ['2025-07-01T00:00:00Z', '2025-07-06T00:00:00Z', '2025-07-26T00:00:00Z' 136, '2025-07-31T00:00:00Z' 136, '2025-08-15T00:00:00Z' 270, '2025-08-30T00:00:00Z', '2025-09-14T00:00:00Z', '2025-09-29T00:00:00Z', '2025-10-14T00:00:00Z', '2025-10-19T00:00:00Z', '2025-11-08T00:00:00Z', '2025-11-18T00:00:00Z']

In [6]:
time_slots = ['2025-07-01T00:00:00Z', '2025-07-06T00:00:00Z', '2025-07-26T00:00:00Z', '2025-07-31T00:00:00Z', '2025-08-15T00:00:00Z', '2025-08-30T00:00:00Z', '2025-09-14T00:00:00Z', '2025-09-29T00:00:00Z', '2025-10-14T00:00:00Z', '2025-10-19T₀:₀₀:₀₀Z', '2０２５-１１-０８T０:００:００Z', '2０２５-１１-１８T０:００:００Z']

In [7]:
best_time_slot = (None, float('inf'), float('inf'))  # (time_slot, missing_edges, reverse_edges)
checked = True
if not checked:
    for time_slot in time_slots:
        ox.settings.overpass_settings = f'[out:json][timeout:60][date:"{time_slot}"]'
        graph = ox.graph_from_bbox(bbox=bbox, network_type='drive')
        nodes, edges = ox.graph_to_gdfs(graph) 
        
        missing = 0
        reverse = 0
        
        for i, edge in edgeinfo.items():
            # Extract u and v, ensuring they are integers (OSMnx expects int IDs)
            u = int(edge[2])
            v = int(edge[3])
            
            if not graph.has_edge(u, v):
                if graph.has_edge(v, u):
                    reverse += 1
                else:
                    missing += 1

        print(f"\n--- Validation Results for time slot {time_slot} ---")
        print(f"Total Edges Checked: {len(edgeinfo)}")
        print(f"Missing Edges: {missing}")
        print(f"Reverse Edges: {reverse}")
        
        if missing < best_time_slot[1] or (missing == best_time_slot[1] and reverse < best_time_slot[2]):
            best_time_slot = (time_slot, missing, reverse)

    print(f"\nBest Time Slot: {best_time_slot[0]} with {best_time_slot[1]} missing edges and {best_time_slot[2]} reverse edges")

In [6]:
is_bbox = True

ox.settings.overpass_settings = f'[out:json][timeout:60][date:"2025-07-10T15:00:00Z"]'

if is_bbox:
    graph = ox.graph_from_bbox(bbox=bbox, network_type='drive')
else:
    
    graph = ox.graph_from_place("Ho Chi Minh City, Vietnam", network_type='drive')
print(f"Network loaded! Total live edges in Ho Chi Minh City: {len(graph.edges)}")  

nodes, edges = ox.graph_to_gdfs(graph) 
edges_metric = edges.to_crs(epsg=3763)

print(f"Historical network loaded! Found {len(edges)} road segments.")

Network loaded! Total live edges in Ho Chi Minh City: 64518
Historical network loaded! Found 64518 road segments.


In [9]:
# valid_edges = []
# invalid_edges = []

# for i, edge in edgeinfo.items():
#     # Extract u and v, ensuring they are integers (OSMnx expects int IDs)
#     u = int(edge[2])
#     v = int(edge[3])
    
#     # Check if this exact directional edge exists in the live Porto graph
#     if graph.has_edge(u, v):
#         valid_edges.append((i, edge))
#     else:
#         # Sometimes an edge exists in the opposite direction (v to u)
#         # We can check for that to diagnose one-way mismatches
#         if graph.has_edge(v, u):
#             invalid_edges.append((i, edge,  "Exists, but opposite direction"))
#         else:
#             invalid_edges.append((i, edge, "Completely missing from graph"))

# print("\n--- Validation Results for ti---")
# print(f"Total Edges Checked: {len(edgeinfo)}")
# print(f"Valid Edges: {len(valid_edges)}")
# print(f"Invalid/Missing Edges: {len(invalid_edges)}")

# # Print a few invalid ones if they exist to investigate
# if invalid_edges:
#     print("\nSample of invalid edges:")
#     for inv_edge in invalid_edges[:5]:
#         print(f"Edge {inv_edge[0]}: {inv_edge[1][2]} -> {inv_edge[1][3]}: {inv_edge[2]}")

In [7]:
our_nodes = gpd.read_file(os.path.join(data_path,"nwk_hcm/hcm_nodes.shp"))
our_edges = gpd.read_file(os.path.join(data_path,"nwk_hcm/hcm_edges.shp"))

In [8]:
our_nodes.head()


,osmid,y,x,street_cou,highway,junction,railway,ref,geometry
0,366367223,10.804243,106.629056,3,None,None,None,None,POINT (106.62906 10.80424)
1,366367322,10.799133,106.657162,3,None,None,None,None,POINT (106.65716 10.79913)
2,366367617,10.810271,106.666859,4,None,None,None,None,POINT (106.66686 10.81027)
3,366367660,10.790142,106.643264,4,None,None,None,None,POINT (106.64326 10.79014)
4,366367822,10.783362,106.628027,4,None,None,None,None,POINT (106.62803 10.78336)


In [9]:
our_edges.head()

,u,v,key,osmid,highway,name,oneway,reversed,length,maxspeed,lanes,junction,bridge,ref,access,tunnel,width,fid,geometry
0,366367223,3771499617,0,607535519,residential,Hẻm 115 Lê Trọng Tấn,F,True,29.493367,None,None,None,None,None,None,None,None,0,"LINESTRING (106.62906 10.80424, 106.62879 10.8..."
1,366367223,366413600,0,32587171,residential,Đường Số 27,F,False,177.327520,None,None,None,None,None,None,None,None,1,"LINESTRING (106.62906 10.80424, 106.62908 10.8..."
2,366367223,4988329156,0,32587171,residential,Đường Số 27,F,True,67.450962,None,None,None,None,None,None,None,None,2,"LINESTRING (106.62906 10.80424, 106.62902 10.8..."
3,366367322,6257994043,0,668267736,residential,Hẻm 97 Nguyễn Thái Bình,F,False,49.470965,None,None,None,None,None,None,None,None,3,"LINESTRING (106.65716 10.79913, 106.65716 10.7..."
4,366367322,11361818084,0,1219459269,tertiary,Nguyễn Thái Bình,F,False,91.724411,None,None,None,None,None,None,None,None,4,"LINESTRING (106.65716 10.79913, 106.65784 10.7..."


In [13]:
# retail_service
# living_service
# catering_service # reverse effect the presence might reduce congestion --> reduce eta
# business_and_company_service
# educational_service
# Medical service
# parking_service # bidirectional, presence might increase congestion --> increase eta

In [10]:
INVALID_TAGS = [
    "traffic_signals",
    "crossing",
    "turning_circle",
    "road",
    "bridge",
    "entrance",
]

In [11]:
columns = ["geometry", "amenity"]

In [16]:
amenity_tags = [
    'mi'
]

In [17]:
POI_MAPPING = {

    # transport
    "bus_station": "transport",
    "subway_entrance": "transport",
    "train_station": "transport",
    "aerodrome": "transport",

    # parking
    "parking": "parking"
}

In [ ]:

tags = {
    "amenity": [
        # catering
        "restaurant",
        "cafe",
        "fast_food",
        "pub",
        "bar",
        "biergarten",
        
        # education
        "school",
        "college",
        "kindergarten",
        "university",

        # medical
        "hospital",
        
        # transport
        "bus_station",
        "taxi",
        
        # parking
        "parking",

    ],
    "public_transport": [
        "station",
        ],
    "aeroway": [
        "aerodrome",
    ],
    "building" : [
         
        # business
        "industrial",
        "office",
        
    ],
    "shop": [
        # living services
        "hairdresser",
        "laundry",
        "beauty",
        
        # retail
        "supermarket",
        "mall",
       
    ]
}
tags = {
    "amenity" : True,
    "shop" : True,
    "aeroway" : True,
    "public_transport" : True,
    "building" : True,
}
gdf_pois = ox.features_from_bbox(bbox=bbox, tags=tags)


In [ ]:
print("Total POIs extracted: ", len(gdf_pois)) 

Total POIs extracted:  56191


In [12]:
tags = {
    "amenity": [
        # catering
        "restaurant", "cafe", "fast_food", "pub", "bar", "biergarten",

        # education
        "school", "college", "kindergarten", "university",

        # medical
        "hospital",

        # transport
        "bus_station", "taxi",

        # parking
        "parking",

        # retail centers
        "shopping_centre",
    ],
    
    "shop": [
        "supermarket", "convenience",
        "hairdresser", "laundry", "beauty",
    ],

    "highway": [
        "bus_stop",
    ],

    "office": True,

    "landuse": [
        "industrial",
    ]
}

In [ ]:
gdf_tags = {}

for tag, value in tags.items():
    try:
        gdf_tag = ox.features_from_bbox(bbox=bbox, tags={tag: value})
        gdf_tags[tag] = gdf_tag
    except Exception as e:
        print(f"{tag} failed:", e)

In [36]:
gdf_pois = pd.concat(gdf_tags.values())

In [38]:
print(gdf_pois.index.names)

['element', 'id']


In [27]:
gdf_pois = gdf_pois.reset_index()

In [29]:
gdf_pois.head()

,element,id,geometry,addr:housenumber,addr:street,amenity,branch,brand,brand:wikidata,cuisine,...,website:vi,old_name:id,building:architecture,building:architecture:typology,name:th,emergency:phone,name:tr,opening_hours:signed,name:ar,industrial
0,node,411918020,POINT (106.6773 10.78245),01,Nguyễn Thông,fast_food,Ga Sài Gòn,Lotteria,Q249525,burger,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,411918246,POINT (106.7001 10.78486),NaN,NaN,fast_food,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,446043808,POINT (106.65726 10.8416),NaN,Đường Lê Văn Thọ,cafe,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,714567078,POINT (106.68985 10.78116),NaN,Ngô Thời Nhiệm,cafe,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,714567113,POINT (106.68993 10.78092),NaN,NaN,cafe,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
gdf_pois_reduced = gdf_pois[gdf_pois.geometry.notnull()]
gdf_pois_reduced = gdf_pois_reduced.set_crs(epsg=4326, allow_override=True)

In [32]:
cols = ["element","id","geometry","amenity","shop","highway","office","landuse"]
gdf_pois_reduced = gdf_pois_reduced[[c for c in cols if c in gdf_pois_reduced.columns]]

In [33]:
gdf_pois_reduced.head()

,element,id,geometry,amenity,shop,highway,office,landuse
0,node,411918020,POINT (106.6773 10.78245),fast_food,NaN,NaN,NaN,NaN
1,node,411918246,POINT (106.7001 10.78486),fast_food,NaN,NaN,NaN,NaN
2,node,446043808,POINT (106.65726 10.8416),cafe,NaN,NaN,NaN,NaN
3,node,714567078,POINT (106.68985 10.78116),cafe,NaN,NaN,NaN,NaN
4,node,714567113,POINT (106.68993 10.78092),cafe,NaN,NaN,NaN,NaN


In [35]:
gdf_pois_reduced['poi_type'] = (
    gdf_pois_reduced['amenity']
    .fillna(gdf_pois_reduced['shop'])
    .fillna(gdf_pois_reduced['highway'])
    .fillna(gdf_pois_reduced['office'])
    .fillna(gdf_pois_reduced['landuse'])
)

gdf_pois_reduced.head()

,element,id,geometry,amenity,shop,highway,office,landuse,poi_type
0,node,411918020,POINT (106.6773 10.78245),fast_food,NaN,NaN,NaN,NaN,fast_food
1,node,411918246,POINT (106.7001 10.78486),fast_food,NaN,NaN,NaN,NaN,fast_food
2,node,446043808,POINT (106.65726 10.8416),cafe,NaN,NaN,NaN,NaN,cafe
3,node,714567078,POINT (106.68985 10.78116),cafe,NaN,NaN,NaN,NaN,cafe
4,node,714567113,POINT (106.68993 10.78092),cafe,NaN,NaN,NaN,NaN,cafe


In [39]:
gdf_pois_reduced.drop(columns=["amenity","shop","highway","office","landuse"], inplace=True)

gdf_pois_reduced.head()

,element,id,geometry,poi_type
0,node,411918020,POINT (106.6773 10.78245),fast_food
1,node,411918246,POINT (106.7001 10.78486),fast_food
2,node,446043808,POINT (106.65726 10.8416),cafe
3,node,714567078,POINT (106.68985 10.78116),cafe
4,node,714567113,POINT (106.68993 10.78092),cafe


In [40]:
gdf_pois_reduced.to_file("gdf_pois_filtered.gpkg", layer="pois", driver="GPKG")

In [ ]:
print(gdf_pois.element.unique())

['node' 'relation' 'way']


In [41]:
POI_MAPPING = {
    "restaurant" : "catering",
    "cafe" : "catering",
    "fast_food" : "catering",
    "pub" : "catering",
    "bar" : "catering",
    "biergarten" : "catering",
    
    "school" : "education",
    "college" : "education",
    "kindergarten" : "education",
    "university" : "education",
    
    "hospital" : "medical",
    
    "bus_station" : "transport",
    "taxi" : "transport",
    "bus_stop" : "transport",
    
    
    "parking" : "parking",
    
    "shopping_centre" : "retail",
    
    "supermarket" : "retail",
    "convenience" : "retail",
    
    "hairdresser" : "living",
    "laundry" : "living",
    "beauty" : "living",
    
    "office" : "business",
    "industrial" : "business",
}

In [43]:
gdf_pois_reduced["poi_group"] = gdf_pois_reduced["poi_type"].map(POI_MAPPING)
gdf_pois_reduced["poi_group"] = gdf_pois_reduced["poi_group"].fillna("other")
gdf_pois_reduced.head()

,element,id,geometry,poi_type,poi_group
0,node,411918020,POINT (106.6773 10.78245),fast_food,catering
1,node,411918246,POINT (106.7001 10.78486),fast_food,catering
2,node,446043808,POINT (106.65726 10.8416),cafe,catering
3,node,714567078,POINT (106.68985 10.78116),cafe,catering
4,node,714567113,POINT (106.68993 10.78092),cafe,catering


In [44]:
gdf_pois_reduced.to_file("gdf_pois_grouping.gpkg", layer="pois", driver="GPKG")

In [ ]:
#TODO map to segment